# Human fibroblast/decidual pseudotime from the final object

Input: the final fibroblast/decidual object. Output: the same object with `obs['pseudotime']`. The procedure is the harmonized time-informed diffusion pseudotime of the Methods (section "Pseudotime"), the same for both species:

1. Couplings between consecutive timepoints with moscot (unbalanced entropic optimal transport in the stromal scVI latent).
2. A CellRank RealTimeKernel combining the couplings with a same-day neighbor graph.
3. A 15-component diffusion map of the symmetrized transition matrix.
4. The root: the earliest-timepoint fibroblast at the fibroblast end of DC1.
5. DPT on all 15 components, rank-normalized.

The code of steps 1 to 5 is that of the runs of record (`harmonized_pseudotime.py`, sha256 `d4b618ae…`; mouse run of 22 Oct 2026, human run of 30 Oct 2026). Variable names are kept so the two can be compared line by line. Only the label and timepoint names were updated to the v9 vocabulary of the final objects.

**Nuclei.** All 47,533 human fibroblast and decidual nuclei are used.

**Embedding.** The pseudotime runs on the 30-dimensional scVI latent fit on these 47,533 nuclei only (30 Oct 2026 refit, which dropped the 471 perivascular nuclei of the earlier stromal latent), with its 3,000 highly variable genes. The human final object does **not** carry that latent: its `obsm['X_scVI']` is the earlier stromal latent. So the latent comes from one of two places (`FIT_LATENT`):

- `False` (default): read the saved latent of record (`LATENT_FILE`, checked against its sha256). CPU only.
- `True`: refit it here from the object's raw counts, with the settings of the run of record. This needs a GPU and takes about 10 minutes on an A40. scVI reproduces only within one GPU class, and the run of record trained on an NVIDIA A40. On another GPU the latent, and therefore the pseudotime, will differ slightly; the check at the end reports by how much.

The latent used is written to the output object as `obsm['X_scVI_fibdec']`. `obsm['X_scVI']` is left as it was.


**Running it.** Run it on the cluster with the `rsc_uv` environment, which has the package versions of the runs of record. Use 8 or more CPUs and about 100 GB of memory; the pseudotime takes about 5 minutes after loading. The notebook reads `INPUT_OBJECT` and writes only `OUTPUT_OBJECT`. Parameters can be set in the next cell or with papermill:

```
papermill pseudotime_human.ipynb pseudotime_human.executed.ipynb -p OUTPUT_OBJECT /path/to/out.h5ad
```

**Output.** A copy of the input object with:

- `obs['pseudotime']`: rank-normalized DPT (average rank / N), 0 at the root and 1 at the end;
- `obsm['X_diffmap']`: the 15 diffusion components, where column 0 is the trivial component and columns 1 and 2 are DC1 and DC2;
- `uns['pseudotime']`: the root, parameters, latent source and versions.

These replace the fields of the same name in the input. The last section compares them with the input's values, which are the run of record.

In [1]:
# Parameters (papermill can override these)
INPUT_OBJECT = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
OUTPUT_OBJECT = "/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_06_pseudotime_notebooks/Cevrim_Human_Only_Fib_Dec_Cells_pseudotime.h5ad"
FIT_LATENT = False                  # True: refit the fib/dec scVI latent from layers['counts'] (GPU)
LATENT_FILE = "/n/lab_storage/eddy_lab/users/nhilgert/tmp/2026_10_30_human_fibdec_latent/run4_a40/fibdec/latent_human_stromal_fibdec.h5ad"
LATENT_SHA256 = "80f3f7eac5fdff9c17ef2b0a5580f5087a9720639704ecd381e481608181b471"
FIT_MAX_EPOCHS = None               # None = scVI's default, as in the run of record; a small number only for testing

## Setup

The package versions are compared with those of the runs of record. Optimal transport and the eigensolver are floating-point computations, so identical results need identical versions, and in human also the same hardware (see the last section).

In [2]:
import datetime
import hashlib
import os
import time
import warnings
from importlib.metadata import version as pkg_version

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")   # as in the runs of record: moscot, CellRank and scanpy warnings are left out

RECORD_VERSIONS = {"scanpy": "1.11.5", "anndata": "0.12.11", "cellrank": "2.0.7", "moscot": "0.5.0",
                   "ott-jax": "0.6.0", "jax": "0.10.0", "numpy": "2.2.0", "scipy": "1.17.1"}
versions = {p: pkg_version(p) for p in RECORD_VERSIONS}
for p, v in versions.items():
    print(f"{p:9s} {v}" + ("" if v == RECORD_VERSIONS[p] else f"   <-- runs of record used {RECORD_VERSIONS[p]}"))

P = dict(epsilon=1e-3, tau=0.95, scale_cost="mean", self_transitions="all", conn_weight=0.2, threshold="auto",
         n_neighbors=15, n_comps=15, n_dcs=15, seed=0)


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()

scanpy    1.11.5
anndata   0.12.11
cellrank  2.0.7
moscot    0.5.0
ott-jax   0.6.0
jax       0.10.0
numpy     2.2.0
scipy     1.17.1


In [3]:
LABEL_COL = "cell_type"
EXCLUDE = {}
N_EXCLUDED = 0
DAY = {"secretory": 0.0, "menstrual": 1.0}
ORDER = ["endometrial_fibroblast", "decidual_cell"]
is_fibroblast = lambda labels: labels == "endometrial_fibroblast"
GENE_SET = "human"
RECORD_ROOT = "CGGGTCATCAGATTGC-1-secretory_mck6c"

## The object

The labels and timepoints are checked against the vocabulary above, so that a renamed category fails here and not silently later.

In [4]:
t_start = time.time()
obj = sc.read_h5ad(INPUT_OBJECT)
input_sha256 = sha256_of(INPUT_OBJECT)
print(f"{INPUT_OBJECT}: {obj.n_obs:,} nuclei x {obj.n_vars:,} genes (sha256 {input_sha256[:12]}...)")
assert obj.n_obs == 47533, obj.n_obs
assert set(obj.obs["timepoint"].astype(str)) == set(DAY), sorted(set(obj.obs["timepoint"].astype(str)))
labels_present = set(obj.obs[LABEL_COL].astype(str))
excluded_labels = {v for values in EXCLUDE.values() for v in values}
assert labels_present == set(ORDER) | excluded_labels, sorted(labels_present)

# The input's own pseudotime and diffusion map (the run of record), kept for the comparison at the end
stored_pseudotime = obj.obs["pseudotime"].copy() if "pseudotime" in obj.obs else None
stored_diffmap = np.asarray(obj.obsm["X_diffmap"]).copy() if "X_diffmap" in obj.obsm else None

/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_Human_Only_Fib_Dec_Cells.h5ad: 47,533 nuclei x 36,601 genes (sha256 e74d038e4460...)


## Embedding: the fibroblast/decidual scVI latent

With `FIT_LATENT = False`, the latent of record is read from `LATENT_FILE` after checking its sha256. With `FIT_LATENT = True`, `fit_fibdec_latent` repeats the 30 Oct 2026 fit (`workflow/scripts/atlas/reintegrate.py` as run for `human_pt_2026_10_30_run4_a40`). It uses raw counts, removes mitochondrial and ribosomal genes, removes genes detected in fewer than 3 nuclei, and selects 3,000 highly variable genes with seurat_v3 on the pooled nuclei. The selection uses no batch key because per-sample variance fits are unstable in small groups; scVI still integrates over `sample`. The model has 30 latent dimensions, 2 layers and a negative-binomial likelihood, with seed 0 and early stopping.

In [5]:
def fit_fibdec_latent(obj, max_epochs=None, seed=0):
    """scVI latent of the fib/dec nuclei with the settings of the 30 Oct 2026 run of record."""
    import scvi
    import torch

    scvi.settings.seed = seed
    np.random.seed(seed)
    adata = ad.AnnData(X=obj.layers["counts"].copy(), obs=obj.obs[["sample"]].copy(), var=pd.DataFrame(index=obj.var_names))
    adata.layers["counts"] = adata.X.copy()
    mito = adata.var_names.str.startswith(("mt-", "Mt-", "MT-"))
    ribo = adata.var_names.str.startswith(("rps", "rpl", "mrps", "mrpl", "Rps", "Rpl", "Mrps", "Mrpl",
                                           "RPS", "RPL", "MRPS", "MRPL"))
    adata_hvg = adata[:, (~mito) & (~ribo)].copy()
    sc.pp.filter_genes(adata_hvg, min_cells=3)
    sc.pp.highly_variable_genes(adata_hvg, flavor="seurat_v3", layer="counts", n_top_genes=3000, subset=True)
    print(f"mito {int(mito.sum())} | ribo {int(ribo.sum())} | {adata_hvg.n_vars} highly variable genes; "
          f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)'}")
    scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
    model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30, gene_likelihood="nb")
    model.train(early_stopping=True, max_epochs=max_epochs)
    return np.asarray(model.get_latent_representation()), pd.Index(adata_hvg.var_names)


if FIT_LATENT:
    latent, hvg_genes = fit_fibdec_latent(obj, max_epochs=FIT_MAX_EPOCHS)
    latent_source = f"refit in this notebook (max_epochs={FIT_MAX_EPOCHS})"
    if os.path.exists(LATENT_FILE):
        # The gene selection does not depend on the GPU, so it should reproduce the record's 3,000 genes exactly.
        lat = ad.read_h5ad(LATENT_FILE, backed="r")
        record_hvg = set(lat.var_names[np.asarray(lat.var["highly_variable"]).astype(bool)])
        lat.file.close()
        print(f"highly variable genes identical to the latent of record: {set(hvg_genes) == record_hvg} "
              f"({len(set(hvg_genes) & record_hvg)} of {len(record_hvg)} shared)")
else:
    assert sha256_of(LATENT_FILE) == LATENT_SHA256, "LATENT_FILE is not the latent of record"
    lat = ad.read_h5ad(LATENT_FILE, backed="r")
    lat_index = pd.Index(lat.obs_names.astype(str)).get_indexer(obj.obs_names.astype(str))
    assert (lat_index >= 0).all(), f"{int((lat_index < 0).sum())} nuclei missing from the latent"
    latent = np.asarray(lat.obsm["X_scVI"])[lat_index]
    hvg_genes = lat.var_names[np.asarray(lat.var["highly_variable"]).astype(bool)]
    lat.file.close()
    latent_source = f"{LATENT_FILE} (sha256 {LATENT_SHA256[:12]}...)"
assert latent.shape == (obj.n_obs, 30) and len(hvg_genes) == 3000, (latent.shape, len(hvg_genes))
print(f"latent {latent.shape[1]} dimensions; {len(hvg_genes)} highly variable genes; source: {latent_source}")

latent 30 dimensions; 3000 highly variable genes; source: /n/lab_storage/eddy_lab/users/nhilgert/tmp/2026_10_30_human_fibdec_latent/run4_a40/fibdec/latent_human_stromal_fibdec.h5ad (sha256 80f3f7eac5fd...)


## Nuclei of the run, timepoints as days, and the working object

The working object `a` holds only what the computation reads: log-normalized expression `.X` (for the same-day PCA graph), the labels, timepoints and samples, the highly variable genes, and the latent as `obsm['X_scVI']`. Fields the input carries from earlier runs (for example `X_diffmap`) are therefore not seen by the steps below.

In [6]:
keep = np.ones(obj.n_obs, dtype=bool)
for col, values in EXCLUDE.items():
    keep &= ~obj.obs[col].astype(str).isin(values).to_numpy()
assert int((~keep).sum()) == N_EXCLUDED, int((~keep).sum())

a = ad.AnnData(X=obj.X[keep].copy(),
               obs=obj.obs.loc[keep, [LABEL_COL, "timepoint", "sample"]].astype(str),
               var=pd.DataFrame(index=obj.var_names))
a.var["highly_variable"] = a.var_names.isin(hvg_genes)       # used by the PCA behind the same-day graph
a.obsm["X_scVI"] = latent[keep]
a.obs["day"] = a.obs.timepoint.map(DAY).astype(float).astype("category")
assert int(a.var.highly_variable.sum()) == 3000
print(f"{obj.n_obs:,} nuclei in the object, {a.n_obs:,} in the run ({N_EXCLUDED} excluded)")
assert a.n_obs == 47533

47,533 nuclei in the object, 47,533 in the run (0 excluded)


## Step 1. Couplings between consecutive timepoints (moscot)

Squared Euclidean cost in the 30-dimensional latent, scaled by its mean; ε = 10⁻³; τa = τb = 0.95; prior growth rates from moscot's proliferation and apoptosis gene sets.

In [7]:
from moscot.problems.time import TemporalProblem
import jax

tp = TemporalProblem(a).score_genes_for_marginals(gene_set_proliferation=GENE_SET, gene_set_apoptosis=GENE_SET)
tp = tp.prepare(time_key="day", joint_attr="X_scVI")
tp = tp.solve(epsilon=P["epsilon"], tau_a=P["tau"], tau_b=P["tau"], scale_cost=P["scale_cost"])
converged = {str(k): bool(pr.solution.converged) for k, pr in tp.problems.items()}
print(f"moscot on {jax.default_backend()}: {len(converged)} coupling(s), all converged = {all(converged.values())}")
assert all(converged.values())

INFO     Solving `1` problems                                                                                      


INFO     Solving problem BirthDeathProblem[stage='prepared', shape=(23441, 24092)].                                


moscot on cpu: 1 coupling(s), all converged = True


## Step 2. CellRank RealTimeKernel

These are the kernel settings of the CellRank protocol: `self_transitions="all"`, `conn_weight=0.2`, `threshold="auto"`. The same-day neighbor graph has 15 neighbors. It is built the scanpy default way, on a 50-component PCA of `.X` restricted to the 3,000 highly variable genes. The row repair is never triggered (0 repaired rows in both runs of record); it is kept so the code matches the script of record.

In [8]:
import cellrank as cr

conn_kwargs = {"n_neighbors": P["n_neighbors"]}                    # no use_rep: the PCA graph of the runs of record
rtk = cr.kernels.RealTimeKernel.from_moscot(tp)
n_dead = 0
try:
    rtk = rtk.compute_transition_matrix(self_transitions=P["self_transitions"], conn_weight=P["conn_weight"],
                                        threshold=P["threshold"], conn_kwargs=conn_kwargs)
except ValueError as e:
    # CellRank rejects a matrix whose rows do not sum to 1; same within-day uniform repair as the released notebooks.
    if "row stochastic" not in str(e):
        raise
    T = csr_matrix(rtk._transition_matrix).tocsr()
    rs = np.asarray(T.sum(1)).ravel()
    dead = np.where(~np.isfinite(rs) | (rs == 0))[0]
    n_dead = len(dead)
    dv = a.obs.day.astype(float).to_numpy()
    L = T.tolil()
    for i in dead:
        same = np.where(dv == dv[i])[0]
        L.rows[i] = list(same)
        L.data[i] = [1.0 / len(same)] * len(same)
    T = L.tocsr()
    T = csr_matrix(T.multiply(1.0 / np.asarray(T.sum(1)).ravel()[:, None]))
    rtk.transition_matrix = T
T = rtk.transition_matrix
dv = a.obs.day.astype(float).to_numpy()
Tc = T.tocoo()
mass = {k: float(Tc.data[np.sign(dv[Tc.col] - dv[Tc.row]) == s].sum() / Tc.data.sum())
        for k, s in (("later", 1), ("same", 0), ("earlier", -1))}
print(f"transition matrix {T.shape}; mass later {mass['later']:.3f} / same {mass['same']:.3f} / "
      f"earlier {mass['earlier']:.3f}; repaired rows {n_dead}")

  0%|          | 0/1 [00:00<?, ?time pair/s]

/n/lab_storage/eddy_lab/users/nhilgert/.uv/envs/rsc_uv/lib/python3.11/site-packages/scanpy/neighbors/__init__.py:577: UserWarning: You’re trying to run this on 36601 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  x = _choose_representation(self._adata, use_rep=use_rep, n_pcs=n_pcs)


/n/lab_storage/eddy_lab/users/nhilgert/.uv/envs/rsc_uv/lib/python3.11/site-packages/scanpy/neighbors/__init__.py:577: UserWarning: You’re trying to run this on 36601 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  x = _choose_representation(self._adata, use_rep=use_rep, n_pcs=n_pcs)


transition matrix (47533, 47533); mass later 0.395 / same 0.605 / earlier 0.000; repaired rows 0


## Step 3. Diffusion map of the symmetrized transition matrix

(T + Tᵀ)/2 with rows renormalized, then the 15 leading eigenvectors, seed 0 (Weiler et al. 2024). Scanpy's `DPT` object needs a neighbor graph to be built, so one is computed on the latent. Its transitions are then replaced by the symmetrized matrix, and the graph itself is not used.

In [9]:
from scanpy.tools._dpt import DPT

S = (T + T.T) / 2
rs = S.sum(axis=1).A1
S.data = S.data / rs[S.nonzero()[0]]
sc.pp.neighbors(a, use_rep="X_scVI", n_neighbors=P["n_neighbors"], random_state=P["seed"])   # required by DPT(); not used
dpt = DPT(adata=a, neighbors_key="neighbors")
dpt._transitions_sym = S
dpt.compute_eigen(n_comps=P["n_comps"], random_state=P["seed"])
a.obsm["X_diffmap"] = dpt.eigen_basis
a.uns["diffmap_evals"] = dpt.eigen_values
DM = np.asarray(a.obsm["X_diffmap"])

## Step 4. Root

The trajectory component is the diffusion component (DC1 to DC5) most correlated with the fibroblast-to-decidual order; it is DC1 in both species. The root is the earliest-timepoint fibroblast at the fibroblast end of that component.

In [10]:
lab = a.obs[LABEL_COL]
order_rank = lab.map({s: i for i, s in enumerate(ORDER)}).to_numpy(float)
assert np.isfinite(order_rank).all(), sorted(set(lab[~np.isfinite(order_rank)]))
rhos = {i: float(spearmanr(DM[:, i], order_rank).statistic) for i in range(1, 6)}
best = max(rhos, key=lambda i: abs(rhos[i]))
fib_end = "max" if rhos[best] < 0 else "min"       # fibroblasts come first in the order, so they sit at the opposite sign
pool = (is_fibroblast(lab) & (a.obs.day.astype(float) == a.obs.day.astype(float).min())).to_numpy()
v = np.where(pool, DM[:, best], np.nan)
root = int(np.nanargmax(v) if fib_end == "max" else np.nanargmin(v))
global_extreme = int(np.argmax(DM[:, best]) if fib_end == "max" else np.argmin(DM[:, best]))
print(f"trajectory component DC{best} (rho {rhos[best]:+.3f}); root {a.obs_names[root]} ({lab.iloc[root]}); "
      f"global extreme {'same cell' if root == global_extreme else a.obs_names[global_extreme]}")

trajectory component DC1 (rho -0.683); root CGGGTCATCAGATTGC-1-secretory_mck6c (endometrial_fibroblast); global extreme same cell


## Step 5. DPT and rank normalization

`scanpy.tl.dpt` on all 15 components of the diffusion map above, rank-normalized as average rank / N over the nuclei of the run.

In [11]:
a.uns["iroot"] = root
sc.tl.dpt(a, n_dcs=P["n_dcs"])
pt = a.obs["dpt_pseudotime"].to_numpy(float)
assert np.isfinite(pt).all() and pt[root] == 0
pt_rank = pd.Series(pt).rank(pct=True).to_numpy()
rho_day = float(spearmanr(pt_rank, a.obs.day.astype(float)).statistic)
print(f"pseudotime on {a.n_obs:,} nuclei; root {a.obs_names[root]}; Spearman rho with timepoint {rho_day:.2f}")

pseudotime on 47,533 nuclei; root CGGGTCATCAGATTGC-1-secretory_mck6c; Spearman rho with timepoint 0.32


## Output object

A copy of the input object, with the pseudotime, the diffusion map and a record of the run. Excluded nuclei, if any, get `NaN`.

In [12]:
out = obj.copy()
pseudotime = pd.Series(np.nan, index=out.obs_names)
pseudotime[a.obs_names] = pt_rank
out.obs["pseudotime"] = pseudotime.to_numpy()
diffmap = np.full((out.n_obs, DM.shape[1]), np.nan)
diffmap[keep] = DM
out.obsm["X_diffmap"] = diffmap
out.obsm["X_scVI_fibdec"] = latent.astype(np.float32)      # the latent the pseudotime was computed on
out.uns["pseudotime"] = dict(
    description="Harmonized time-informed diffusion pseudotime (moscot couplings, CellRank RealTimeKernel, "
                "symmetrized diffusion map, DPT), rank-normalized as average rank / N; see Methods, Pseudotime.",
    root=str(a.obs_names[root]), root_label=str(lab.iloc[root]), trajectory_dc=int(best), n_cells=int(a.n_obs),
    n_dcs=P["n_dcs"], params={k: str(v) for k, v in P.items()}, latent=latent_source,
    diffmap_evals=np.asarray(a.uns["diffmap_evals"], dtype=float), transition_mass={k: float(v) for k, v in mass.items()},
    input_object=INPUT_OBJECT, input_sha256=input_sha256, versions=versions,
    computed=datetime.datetime.now().isoformat(timespec="seconds"))

os.makedirs(os.path.dirname(OUTPUT_OBJECT), exist_ok=True)
out.write_h5ad(OUTPUT_OBJECT, compression="gzip")
print(f"wrote {OUTPUT_OBJECT} ({out.n_obs:,} nuclei; {int(out.obs['pseudotime'].notna().sum()):,} with pseudotime) "
      f"in {time.time() - t_start:.0f} s")

wrote /n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_06_pseudotime_notebooks/Cevrim_Human_Only_Fib_Dec_Cells_pseudotime.h5ad (47,533 nuclei; 47,533 with pseudotime) in 171 s


## Comparison with the pseudotime of record

The input object's `obs['pseudotime']` and `obsm['X_diffmap']` are the runs of record. On the same software, the mouse computation has reproduced them exactly on Intel nodes with 8 CPUs, and to within 4 × 10⁻⁵ in rank pseudotime on an AMD node. The human run of record was computed on one particular node (holygpu7c0530: Intel Cascade Lake, 16 CPUs, JAX on CPU), and human reruns on other nodes have differed slightly through the order of floating-point additions: same root, ρ ≈ 0.999 with the record, and no reported number changed. A latent refit on a GPU other than an A40 will differ more. The numbers below show which case applies.

In [13]:
if stored_pseudotime is None:
    print("The input object has no pseudotime to compare with.")
else:
    old = stored_pseudotime.to_numpy(float)
    new = out.obs["pseudotime"].to_numpy(float)
    same_nan = bool(np.array_equal(np.isnan(old), np.isnan(new)))
    both = ~np.isnan(old) & ~np.isnan(new)
    diff = np.abs(new[both] - old[both])
    dm_old, dm_new = stored_diffmap[both], out.obsm["X_diffmap"][both]
    print(f"nuclei without pseudotime: same set {same_nan} ({int(np.isnan(new).sum())})")
    print(f"pseudotime identical to the record: {bool(np.array_equal(new[both], old[both]))}")
    print(f"  Spearman rho with the record {spearmanr(new[both], old[both]).statistic:.6f}; |difference| median "
          f"{np.median(diff):.2e}, 99th percentile {np.quantile(diff, 0.99):.2e}, largest {diff.max():.2e}; "
          f"nuclei moved by more than 0.05: {int((diff > 0.05).sum())}")
    print(f"diffusion map identical to the record: {bool(np.array_equal(dm_new, dm_old))} "
          f"(largest difference {np.nanmax(np.abs(dm_new - dm_old)):.2e})")
    print(f"root: {out.uns['pseudotime']['root']} (Methods: {RECORD_ROOT}); same {out.uns['pseudotime']['root'] == RECORD_ROOT}")

nuclei without pseudotime: same set True (0)
pseudotime identical to the record: False
  Spearman rho with the record 0.999994; |difference| median 2.52e-04, 99th percentile 2.61e-03, largest 7.21e-02; nuclei moved by more than 0.05: 2
diffusion map identical to the record: False (largest difference 1.56e-03)
root: CGGGTCATCAGATTGC-1-secretory_mck6c (Methods: CGGGTCATCAGATTGC-1-secretory_mck6c); same True
